In [1]:
from langgraph.graph import StateGraph,START, END
from typing import TypedDict, List,Literal
from dotenv import load_dotenv
from pydantic import BaseModel,Field
from langchain_google_genai import ChatGoogleGenerativeAI
import google.genai as genai

load_dotenv()
import os

c:\Users\H604660\Desktop\Siddhi Shintre\My Learnings\LANGGRAPH\myenv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [3]:
class SentimentSchema(BaseModel):
    sentiment : Literal["positive","negative"] = Field(description="The sentiment of the review")

In [23]:
class DiagnosisSchema(BaseModel):
    issues : Literal['UX','Performance',"Bug","Support","Other"] = Field(description="List of issues mentioned in the review")
    tone : Literal['Angry','Frustrated','Neutral','Confused','Disappointed'] = Field(description
    ="Tone of the customer in the review")
    urgency : Literal['High','Medium','Low'] = Field(description="Urgency level of the customer's issues")

In [11]:
prompt = "What is sentiment of the review - The software too good "

In [ ]:
client = genai.Client(
    vertexai=True, 
    project=os.getenv('GEMINI_API_KEY'), 
    location='us-central1'
)

GEMINI_SETTINGS = {
    "model": "gemini-2.0-flash",
    "config": {
        'response_mime_type': 'application/json',
        'response_schema': SentimentSchema,
    }
}

GEMINI_SETTINGS1 = {
    "model": "gemini-2.0-flash",
    "config": {
        'response_mime_type': 'application/json',
        'response_schema': DiagnosisSchema,
    }
}



In [13]:
response = client.models.generate_content(
    contents=prompt,
    **GEMINI_SETTINGS
)

c:\Users\H604660\Desktop\Siddhi Shintre\My Learnings\LANGGRAPH\myenv\Lib\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [14]:
structured_data = response.parsed.sentiment
print(structured_data)

positive


In [15]:
class ReviewState(TypedDict):
    review : str
    sentiment : Literal["positive","negative"]
    diagnosis : dict
    response : str

In [16]:
def find_sentiment(state: ReviewState):
    prompt = f"What is sentiment of the review - {state['review']}"
    sentiment = client.models.generate_content(
    contents=prompt,
    **GEMINI_SETTINGS
    ).parsed.sentiment

    return {'sentiment' : sentiment}

In [28]:
def check_sentiment(state : ReviewState)-> Literal['positive_response','run_diagnosis']:
    if state['sentiment'] == 'positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [50]:
def positive_response(state:ReviewState):
    prompt = f"""Write a  warm thank you response to the positive review - {state['review']} \n
    also ask the customer to share their experience on social media."""

    response = client.models.generate_content(
    model="gemini-2.0-flash",    
    contents=prompt
    )

    return {'response' : response.text}

In [51]:
#HERE  OUTPUT OF DIAGNOSIS IS A DICTIONARY WITH ISSUES,TONE AND URGENCY KEYS


def run_diagnosis(state:ReviewState):
    prompt = f"""Analyze the following review and extract the issues mentioned, tone of the customer and urgency level - {state['review']} \n
    Provide the response in JSON format with keys as issues,tone and urgency."""

    diagnosis = client.models.generate_content(
    contents=prompt,
    **GEMINI_SETTINGS1
    ).parsed

    return {'diagnosis' : diagnosis.dict()}

In [68]:
def negative_response(state:ReviewState):
    diagnosis = state['diagnosis']
    prompt = f"""  You are a support assistant. 
      The use had a  {diagnosis['issues']} issue with {diagnosis['tone']} tone and {diagnosis['urgency']} urgency.
      Write a empathetic response addressing the user's concerns based on the review """

    response = client.models.generate_content(
    contents=prompt,
    model="gemini-2.0-flash"

    )

    return {'response' : response.text}

In [69]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('negative_response',negative_response)
    
graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)

graph.add_edge('positive_response',END)

graph.add_edge('run_diagnosis','negative_response')

graph.add_edge('negative_response',END)

In [70]:
workflow = graph.compile()

In [71]:
#Displaying the graph
print(workflow.get_graph().draw_ascii())

                    +-----------+                      
                    | __start__ |                      
                    +-----------+                      
                          *                            
                          *                            
                          *                            
                  +----------------+                   
                  | find_sentiment |                   
                  +----------------+                   
                  ..             ...                   
               ...                  ...                
             ..                        ..              
  +---------------+                      ..            
  | run_diagnosis |                       .            
  +---------------+                       .            
          *                               .            
          *                               .            
          *                               .     

In [72]:
initial_state = {
    'review' : "the product is really good"
}

In [73]:
workflow.invoke(initial_state)

{'review': 'the product is really good',
 'sentiment': 'positive',
 'response': '"Thank you so much for your wonderful review! We\'re thrilled to hear you\'re enjoying our [Product Name]. It truly makes our day to know that our product is meeting and exceeding your expectations. We put a lot of heart into [mention a key feature or aspect of the product], and it\'s fantastic to hear that it\'s resonating with you.\n\nWe\'d love it if you\'d be willing to share your experience with your friends and followers on social media! A quick post about what you love about [Product Name], maybe with a photo or video, would mean the world to us. You can tag us [@YourSocialMediaHandle] so we can see it and share it too!\n\nThanks again for your support!"\n'}

In [74]:
initial_state1 = {
    'review' : " I have been using  this app  for about a month now and I am extremely disappointed with its performance. The app crashes frequently, causing me to lose important data. Additionally, the user interface is not intuitive, making it difficult to navigate through the features. I expected a much better experience given the high ratings and positive reviews. This has been a frustrating experience overall, and I hope the developers address these issues soon."
}
workflow.invoke(initial_state1)

C:\Users\H604660\AppData\Local\Temp\ipykernel_29044\677859599.py:13: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return {'diagnosis' : diagnosis.dict()}


{'review': ' I have been using  this app  for about a month now and I am extremely disappointed with its performance. The app crashes frequently, causing me to lose important data. Additionally, the user interface is not intuitive, making it difficult to navigate through the features. I expected a much better experience given the high ratings and positive reviews. This has been a frustrating experience overall, and I hope the developers address these issues soon.',
 'sentiment': 'negative',
 'diagnosis': {'issues': 'UX', 'tone': 'Frustrated', 'urgency': 'High'},
 'response': 'Okay, I\'m ready. Please provide me with the user\'s review or describe the UX issue they experienced, including the details that convey their frustration and urgency. The more information I have, the better I can craft an empathetic and helpful response.\n\nFor example, tell me:\n\n*   **What was the user trying to do?**\n*   **What went wrong?**\n*   **What specific language did they use to express their frustra